# 09 — V3-B Recall-First Hyperparameter Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shelly-serafimovich/target-aware-infrared-sr/blob/main/notebooks/09_v3b_recall_tuning.ipynb)


Tunes only V3-B on the exact same frozen 280-image pilot manifest from Notebook 08.

Selection is recall-first: maximize small-target GT-matched recall (≤24×24, IoU≥0.25), then mean GT confidence, while monitoring background false alarms.


In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
from pathlib import Path
import os,sys,subprocess,json,re,hashlib,time
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
from PIL import Image

PROJECT_REPO='https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
DIFIISR_REPO='https://github.com/zirui0625/DifIISR.git'
DIFIISR_COMMIT='09ca97ea48d481656dd8090e84099c963059ac41'
SEED=12345

DATASET_ROOT=Path('/content/drive/MyDrive/thermal-drone-dataset')
LR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'LR_difiisr_train_degradation_v2'
YOLO_WEIGHTS=DATASET_ROOT/'detector'/'weights'/'thermal_drone_yolo11n_corrected_v1_best.pt'
EXPERIMENT_ROOT=DATASET_ROOT/'target_aware_difiisr'/'v3_recall_pilot_v1'
EXPERIMENT_ROOT.mkdir(parents=True,exist_ok=True)
project_dir=Path('/content/target-aware-infrared-sr'); difiisr_dir=Path('/content/DifIISR')

def run(cmd,cwd=None):
    print('$',' '.join(map(str,cmd)),flush=True); subprocess.run(list(map(str,cmd)),cwd=cwd,check=True)

if not project_dir.exists(): run(['git','clone',PROJECT_REPO,project_dir])
else:
    run(['git','fetch','origin'],cwd=project_dir); run(['git','checkout','main'],cwd=project_dir); run(['git','pull','--ff-only','origin','main'],cwd=project_dir)
if not difiisr_dir.exists(): run(['git','clone',DIFIISR_REPO,difiisr_dir])
run(['git','fetch','--all'],cwd=difiisr_dir); run(['git','checkout','--detach',DIFIISR_COMMIT],cwd=difiisr_dir)

ANN_DIR=project_dir/'annotations'/'corrected_yolo'/'valid'/'labels'
ENV=Path('/content/difiisr-env'); PY=str(ENV/'bin'/'python')
if not Path(PY).exists():
    run([sys.executable,'-m','pip','install','-q','uv'])
    run([sys.executable,'-m','uv','venv','--python','3.10',ENV])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'torch==2.1.1','torchvision==0.16.1','xformers==0.0.23','numpy==1.25.2','scipy==1.9.3','packaging==24.2','setuptools==69.5.1'])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'basicsr==1.4.2','opencv-python','matplotlib','timm','pandas','pillow','scikit-learn','scikit-image','lpips','loguru','omegaconf','six','tqdm','albumentations','einops','imageio','gdown'])
run([sys.executable,'-m','pip','install','-q','ultralytics'])

weights=difiisr_dir/'weights'; weights.mkdir(exist_ok=True)
if not (weights/'DifIISR.pth').exists(): run([PY,'-m','gdown','1PhRvk1Dlp3CCrPkrxRfNbNZ3fNgviDVZ','-O',weights/'DifIISR.pth'])
if not (weights/'autoencoder_vq_f4.pth').exists(): run([PY,'-m','gdown','https://github.com/zsyOAOA/ResShift/releases/download/v2.0/autoencoder_vq_f4.pth','-O',weights/'autoencoder_vq_f4.pth'])

assert LR_DIR.exists() and ANN_DIR.exists() and YOLO_WEIGHTS.exists()
print('✓ setup ready | GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')


In [ ]:
# Reuse the exact Notebook 08 pilot manifest; do not resample validation images.
PILOT_N_TARGET_IMAGES=200; PILOT_N_BACKGROUND_IMAGES=80
MANIFEST_PATH=EXPERIMENT_ROOT/f'pilot_manifest_{PILOT_N_TARGET_IMAGES}t_{PILOT_N_BACKGROUND_IMAGES}b_seed{SEED}.csv'
assert MANIFEST_PATH.exists(), f'Run Notebook 08 once first; missing {MANIFEST_PATH}'
print('Using fixed pilot:',MANIFEST_PATH)


In [ ]:
# Focused V3-B sweep around the successful Notebook 08 setting.
COMMON=dict(seed=SEED,pilot_n_target_images=200,pilot_n_background_images=80,
    small_target_definition='upsampled pre-encoder bbox width<=24 AND height<=24',
    core=24,outer=48,top_k=15,sigma_floor=0.013458,blob_k=1.5,soft_t=0.25,soft_min_support=3.0,
    scr_gate=1.8,scr_gate_temp=0.25,nd_gate=0.40,nd_gate_temp=0.07,nd_floor_weight=0.50,
    max_candidates=48,proposal_nms=7,scr_goal=2.5,goal_temp=0.25,lambda_fid=10.0,lambda_bg=20.0,
    difiisr_commit=DIFIISR_COMMIT,primary_selection_metric='GT-matched recall @ IoU>=0.25 on <=24x24 targets')

SWEEP=[('s2_lr003',2,0.03),('s3_lr002',3,0.02),('s3_lr003',3,0.03),('s4_lr002',4,0.02)]
display(pd.DataFrame(SWEEP,columns=['name','steps','lr']))


In [ ]:
# Generate/resume every configuration in a separate run folder.
RUNNER=project_dir/'scripts'/'v3_recall_guidance_runner.py'
DIFIISR_CONFIG=difiisr_dir/'configs'/'DifIISR_test.yaml'
TUNING_ROOT=DATASET_ROOT/'target_aware_difiisr'/'v3b_recall_tuning_v1'; TUNING_ROOT.mkdir(parents=True,exist_ok=True)
runs=[]
for label,steps,lr in SWEEP:
    cfg=dict(COMMON,guidance_steps=steps,guidance_lr=lr)
    run_id='cfg_'+hashlib.sha1(json.dumps(cfg,sort_keys=True,separators=(',',':')).encode()).hexdigest()[:10]
    run_root=TUNING_ROOT/run_id; run_root.mkdir(parents=True,exist_ok=True)
    cfg_path=run_root/'config.json'; cfg_path.write_text(json.dumps(cfg,indent=2))
    print('\n'+'='*90); print(label,run_id,'steps=',steps,'lr=',lr); print('='*90)
    cmd=[PY,str(RUNNER),'--manifest',str(MANIFEST_PATH),'--lr',str(LR_DIR),'--out',str(run_root),'--config',str(cfg_path),'--difiisr-config',str(DIFIISR_CONFIG)]
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout: print(line,end='',flush=True)
    rc=proc.wait()
    if rc: raise RuntimeError(f'{label} generation failed ({rc}); completed images remain safe in {run_root}')
    runs.append((label,steps,lr,run_id,run_root))
print('✓ sweep generation complete')


In [ ]:
# Frozen-YOLO evaluation for every configuration.
EVAL_SCRIPT=project_dir/'scripts'/'v3_recall_yolo_eval.py'
for label,steps,lr,run_id,run_root in runs:
    print('\nYOLO EVAL:',label)
    cmd=[sys.executable,str(EVAL_SCRIPT),'--manifest',str(MANIFEST_PATH),'--run-root',str(run_root),'--weights',str(YOLO_WEIGHTS),'--core','24','--iou-match','0.25','--fa-conf','0.25']
    subprocess.run(cmd,check=True)
print('✓ all YOLO evaluations complete')


In [ ]:
# Recall-first leaderboard.
rows=[]
for label,steps,lr,run_id,run_root in runs:
    er=run_root/'yolo_pilot_eval'; sm=pd.read_csv(er/'pilot_summary.csv'); pr=pd.read_csv(er/'paired_saved_lost.csv')
    b=sm[sm.method=='V3-B'].iloc[0]; pp=pr[pr.method=='V3-B'].iloc[0]
    rows.append(dict(config=label,steps=steps,lr=lr,recall=b.small_target_recall_iou025,recall_delta_pp=b.recall_delta_pp_vs_BASE,
        mean_gt_conf=b.mean_gt_conf,saved=int(pp.saved),lost=int(pp.lost),net_saved=int(pp.net_saved),
        bg_images_with_FA=int(b.background_images_with_FA_conf025),bg_FA_count=int(b.background_FA_count_conf025),bg_max_conf_mean=b.background_max_conf_mean,run_id=run_id))
leader=pd.DataFrame(rows).sort_values(['recall','mean_gt_conf'],ascending=[False,False]).reset_index(drop=True)
display(leader)
best=leader.iloc[0]
print('\nBEST (recall first):',best['config'],'recall=',best['recall'],'delta pp=',best['recall_delta_pp'])
print('saved/lost=',int(best['saved']),int(best['lost']),'| background FA count=',int(best['bg_FA_count']))
leader.to_csv(TUNING_ROOT/'leaderboard.csv',index=False)
print('Saved:',TUNING_ROOT/'leaderboard.csv')


## Decision
Do not automatically promote the winner to the full validation set. First inspect the leaderboard. We want a repeatable recall gain over the Notebook 08 V3-B result (19.0% vs BASE 18.5%) without a large false-alarm increase.
